In [ ]:
!pip install --upgrade scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 107.7 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [ ]:
import xgboost as xgb
import shap
import pandas as pd
import numpy as np
from typing import Union, Dict, Optional, Tuple, Set, List
from math import factorial
import time
from copy import copy
from tqdm import tqdm
from collections import defaultdict
from sklearn.metrics import accuracy_score, f1_score
import sklearn
import math
import cupy as cp

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module=r"sklearn\..*")

In [ ]:
# Useful if you run this on google colab and downloaded the data into your drive.
# If you run the notebook in other environment remove these lines and change the 'pd.read_csv()' function in this notebook to read from
# where you saved you data
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# import woodelf from Python file in the drive
!cp /content/drive/MyDrive/...../woodelf.py /content/

import woodelf

# PDP Code

In [ ]:
class CPDVMetric(woodelf.CubeMetric):
    def calc_metric(self, s_plus: Set, s_minus: Set) -> Dict[str, float]:
        if len(s_plus & s_minus) > 0:
            return {}
        pdp_values = {}
        if len(s_plus) == 1:
            for f in s_plus:
                pdp_values[f] = 1
        if len(s_plus) == 0:
            for f in s_minus:
                pdp_values[f] = -1
        return pdp_values

In [ ]:
class PathToValuesMatrixLimitSPlus(woodelf.PathToValuesMatrix):
    # Ignored all cubes with |S^+| > MAX_S_PLUS_SIZE to reduce the complexity element of TL3**D to TL2**D*(D**MAX_S_PLUS_SIZE)
    MAX_S_PLUS_SIZE = NotImplemented

    @classmethod
    def map_patterns_to_cube(cls, features_in_path: List[str]):
        updated_wdnf_table = {0: {0: (set(), set())}}
        current_wdnf_table = None
        for feature in features_in_path:
            current_wdnf_table = updated_wdnf_table
            updated_wdnf_table = {}
            for consumer_pattern in current_wdnf_table:
                updated_wdnf_table[consumer_pattern * 2 + 0] = {}
                updated_wdnf_table[consumer_pattern * 2 + 1] = {}
                for background_pattern in current_wdnf_table[consumer_pattern]:
                    s_plus, s_minus = current_wdnf_table[consumer_pattern][background_pattern]

                    # The implementation is identical to the PathToValuesMatrix.map_patterns_to_cube implementation, except for this if.
                    if len(s_plus | {feature}) <= cls.MAX_S_PLUS_SIZE:
                        updated_wdnf_table[consumer_pattern * 2 + 1][background_pattern * 2 + 0] = (s_plus | {feature}, s_minus) # Rule 1

                    updated_wdnf_table[consumer_pattern * 2 + 0][background_pattern * 2 + 1] = (s_plus, s_minus | {feature}) # Rule 2
                    updated_wdnf_table[consumer_pattern * 2 + 1][background_pattern * 2 + 1] = (s_plus, s_minus) # Rule 3

        return updated_wdnf_table

class PathToValuesMatrixLimitSPlusTo1(PathToValuesMatrixLimitSPlus):
    # We uses the fact CPDVMetric ignored all cubes with |S^+| > 1 to reduce the complexity element of TL3**D to TL2**D*D
    MAX_S_PLUS_SIZE = 1

class PathToValuesMatrixLimitSPlusTo2(PathToValuesMatrixLimitSPlus):
    # We uses the fact PDIVOrder1Or2 ignored all cubes with |S^+| > 2 to reduce the complexity element of TL3**D to TL(2**D)*(D**2)
    MAX_S_PLUS_SIZE = 2

In [ ]:
def build_sampled_points_df(data: pd.DataFrame, k: int, seed: int = None):
    """
    Sample k points from every column.
    """
    sample_points_data = {}
    for f in data.columns:
        sample_points_data[f] = list(data[f].sample(k, random_state=seed))
        sample_points_data[f].sort()
    return pd.DataFrame(sample_points_data)[data.columns]

def build_equally_distanced_points_df(data: pd.DataFrame, k: int, percentiles: Tuple[float]):
    """
    Take equally distanced points from each column. The min point will be in the precentile percentiles[0]
    and the max point will be in the precentile percentiles[1].
    This is also the default implementation of sklearn
    """
    sample_points_data = {}
    for f in data.columns:
        low, high = np.percentile(data[f].dropna(), [percentiles[0] * 100, percentiles[1]*100])
        # get k equally spaced points between them
        points = np.linspace(low, high, k)
        sample_points_data[f] = list(points)
        sample_points_data[f].sort()
    return pd.DataFrame(sample_points_data)[data.columns]

def build_points_for_full_pdp(data: pd.DataFrame, model, as_df: bool=True):
    """
    Provide the points that will create a full PDP - a graph the will provide the PDV for every x value.
    Does this by collecting all the threshold values from the model. See Sect. of the paper.
    """
    # load the model
    model_objs = woodelf.load_decision_tree_ensamble_model(model, list(data.columns))

    # collect all the theshold values for each feature
    th_values = {f: [] for f in list(data.columns)}
    for tree in model_objs:
        for node in tree.bfs(including_myself=True, including_leaves=False):
            th_values[node.feature_name].append(node.value)

    # Make sure the thesholds are unique and sort them
    for f in th_values:
        th_values[f] = sorted(list(set(th_values[f])))

    if not as_df:
        return th_values

    # zfill
    max_th_length = max([len(thersholds) for thersholds in th_values.values()])
    for f in th_values:
        th_values[f].extend([0] * (max_th_length - len(th_values[f])) )

    # from the built thershold build the points Data Frame
    return pd.DataFrame(th_values)

In [ ]:
def build_points_for_pdp(model, data: pd.DataFrame, k: int = 100, percentiles: Tuple[float] = (0.05, 0.95), sampled: bool = False, seed: int = 42, full_pdp: bool = False, verbose : bool = True):
    start_time = time.time()
    if sampled:
        points_df = build_sampled_points_df(data, k, seed)
    elif full_pdp:
        points_df = build_points_for_full_pdp(data, model)
    else:
        points_df = build_equally_distanced_points_df(data, k, percentiles)
    if verbose:
        print(f"Building the points took: {time.time() - start_time} sec")
    return points_df

def woodelf_pdp(model, data: pd.DataFrame, k: int = 100, accurate: bool = True, centered: bool = True, GPU: bool = False,
                percentiles: Tuple[float] = (0.05, 0.95), sampled: bool = False, seed: int = 42, full_pdp: bool = False):
    """
    Compute all the PDVs needed in order to plot the PDP values of all the features. Use WOODELF!
    """
    points_df = build_points_for_pdp(model, data, k, percentiles, sampled, seed, full_pdp, verbose=True)
    return woodelf_pdp_given_points_df(model, data, points_df, accurate, centered, GPU), points_df

def woodelf_pdp_given_points_df(model, data: pd.DataFrame, sampled_points_df: pd.DataFrame, accurate: bool = True, centered: bool = True, GPU: bool = False):
    """
    Compute all the PDVs of the provided points. Use WOODELF!
    """
    metric=CPDVMetric()
    p2v = PathToValuesMatrixLimitSPlusTo1(metric)
    if accurate:
        pdvs = woodelf.calculate_background_metric(model, consumer_data=sampled_points_df, background_data=data, metric=metric, global_importance=False, GPU=GPU, path_to_matrixes_calculator=p2v)
    else:
        pdvs = woodelf.calculate_path_dependent_metric(model, consumer_data=sampled_points_df, metric=metric, global_importance=False, GPU=GPU, path_to_matrixes_calculator=p2v)
    if centered:
        return pdvs

    avg_prediction = float(model.predict(data).mean())
    for f in pdvs:
        pdvs[f] += avg_prediction
    return pdvs

## Joint DPD code

In [ ]:
# PDP joint

from itertools import combinations

def all_subsets_of_size_0_1_2(s):
    subsets = [set()]
    for k in [1,2]:
        for subset in combinations(s, k):
            subsets.append(set(subset))
    return subsets

class PDIVOrder1Or2(woodelf.CubeMetric):
    INTERACTION_VALUE = True

    def calc_metric(self, s_plus: Set, s_minus: Set) -> Dict[str, float]:
        if len(s_plus & s_minus) > 0:
            return {}

        pdivs = {}
        for sm in all_subsets_of_size_0_1_2(s_minus):
            s = tuple(s_plus | sm)
            if len(s) in [1,2]:
                pdivs[s] = (-1) ** (len(sm))
        return pdivs

def bits(n, D):
    bs = []
    for i in range(D):
        bs.append(n % 2)
        n = n // 2
    return reversed(bs)

def build_points_for_joint_pdp(points_df: pd.DataFrame):
    D = math.ceil(math.log2(len(points_df.columns)))
    data = {f: [] for f in points_df.columns}
    k = len(points_df)
    for i, f in enumerate(points_df.columns):
        for b in bits(i, D):
            if b == 0:
                data[f].extend(np.tile(points_df[f].values, k))
            elif b == 1:
                data[f].extend(np.repeat(points_df[f].values, k))
    return pd.DataFrame(data)

def first_different_bit(n1, n2, D):
    assert n1 != n2
    i = 0
    for b1, b2 in zip(bits(n1, D), bits(n2, D)):
        if b1 != b2:
            return i
        i += 1

def clip_result(pdvs, features, k):
    D = math.ceil(math.log2(len(features)))
    feature_to_index = {f:i for i,f in enumerate(features)}
    clipped = {}
    for f1, f2 in pdvs:
        i1 = feature_to_index[f1]
        i2 = feature_to_index[f2]
        h = first_different_bit(i1, i2, D)
        clipped[(f1, f2)] = pdvs[(f1, f2)][h*(k**2): (h+1)*(k**2)]
    return clipped


def woodelf_pdp_joint(model, data: pd.DataFrame, k: int = 100, accurate: bool = True, centered: bool = True, GPU: bool = False,
                percentiles: Tuple[float] = (0.05, 0.95), sampled: bool = False, seed: int = 42, full_pdp: bool = False, verbose: bool = True):
    """
    Compute all the PDVs needed in order to plot the PDP values of all the features. Use WOODELF!
    """
    start_time = time.time()
    original_points_df = build_points_for_pdp(model, data, k, percentiles, sampled, seed, full_pdp, verbose=False)
    if full_pdp:
        k = len(original_points_df)

    points_df = build_points_for_joint_pdp(original_points_df)
    if verbose:
        print(f"Building the points took: {time.time() - start_time} sec. The size of the created df {len(points_df)}")

    metric = PDIVOrder1Or2()
    p2v = PathToValuesMatrixLimitSPlusTo2(metric)
    if accurate:
        pdivs = woodelf.calculate_background_metric(
            model, consumer_data=points_df, background_data=data, metric=metric, global_importance=False, GPU=GPU, path_to_matrixes_calculator=p2v
        )
    else:
        pdivs = woodelf.calculate_path_dependent_metric(
            model, consumer_data=points_df, metric=metric, global_importance=False, GPU=GPU, path_to_matrixes_calculator=p2v
        )
    avg_prediction = float(model.predict(data).mean())
    if GPU:
        base_pdv = cp.array([avg_prediction] * len(points_df))
        zero_array = cp.array([0] * len(points_df))
    else:
        base_pdv = np.array([avg_prediction] * len(points_df))
        zero_array = np.array([0] * len(points_df))
    pdvs = {}

    D = math.ceil(math.log2(len(points_df.columns)))
    points_parts = {f: [points_df[f].values[i:i + k**2] for i in range(0, len(points_df[f]), k**2)] for f in data.columns}
    f1_points = {}
    f2_points = {}
    for i, f1 in enumerate(data.columns):
        for j, f2 in enumerate(data.columns):
            if f1 != f2:
                pair = (f1, f2)
                pdvs[(f1,f2)] = base_pdv + pdivs.get((f1,), zero_array) + pdivs.get((f2,), zero_array) + pdivs.get(pair, zero_array)
                points_part_index = first_different_bit(i,j,D)
                f1_points[(f1,f2)] = points_parts[f1][points_part_index]
                f2_points[(f1,f2)] = points_parts[f2][points_part_index]
    clipped_pdvs = clip_result(pdvs, list(data.columns), k)
    return clipped_pdvs, f1_points, f2_points

## Any Order PDIV code

In [ ]:
from itertools import combinations

def all_subsets(s):
    subsets = []
    for k in range(len(s) + 1):
        for subset in combinations(s, k):
            subsets.append(set(subset))
    return subsets

class PDIV(woodelf.CubeMetric):
    INTERACTION_VALUE = True
    def calc_metric(
        self, s_plus: Set, s_minus: Set
    ) -> Dict[str, float]:
        if len(s_plus & s_minus) > 0:
            return {}

        pdivs = {}
        for sm in all_subsets(s_minus):
            s = tuple(s_plus | sm)
            pdivs[s] = (-1) ** (len(sm))
        return pdivs

# Fraud Data + Model

In [ ]:
transactions_train = pd.read_parquet('drive/MyDrive/PATH_TO_DATA_CREATED_IN_THE_Preprocess_Data_NOTEBOOK.parquet') # columns are train_features + ['isFraud']
transactions_test = pd.read_parquet('drive/MyDrive/PATH_TO_DATA_CREATED_IN_THE_Preprocess_Data_NOTEBOOK.parquet') # columns are train_features + ['isFraud']

train_features = [f for f in transactions_train.columns if f != 'isFraud']
fraud_train = transactions_train[train_features]
fraud_test = transactions_test[train_features]

In [ ]:
gradient_boosting_model = sklearn.ensemble.HistGradientBoostingRegressor(
    max_iter=100,
    max_depth=6,
    max_leaf_nodes=None,
    # tree_method="hist",  # use 'gpu_hist' if you want GPU support
    random_state=42
)
gradient_boosting_model.fit(fraud_train, transactions_train['isFraud'])

y_pred = gradient_boosting_model.predict(transactions_test[train_features])
print(f"Accuracy: {accuracy_score(transactions_test['isFraud'], y_pred.round())}, F1 score: {f1_score(transactions_test['isFraud'], y_pred.round())}")

Accuracy: 0.9656246824939886, F1 score: 0.365625


## Woodelf PDP computation



In [ ]:
def get_pdp_testing_params():
    return {
        "Exact PDP k=5":      dict(k = 5,         accurate = True,  centered = False, GPU = True),
        "Exact PDP k=10":     dict(k = 10,        accurate = True,  centered = False, GPU = True),
        "Exact PDP k=100":    dict(k = 100,       accurate = True,  centered = False, GPU = True),
        "Exact full PDP":     dict(full_pdp=True, accurate = True,  centered = False, GPU = True),
        "Estimated PDP k=5":  dict(k = 5,         accurate = False, centered = True,  GPU = True),
    }

def messure_woodelf_pdp_times(model, data, params):
    running_results = {}
    for name, running_params in params.items():
        print(name + ":")
        start_time = time.time()
        woodelf_pdp(model, data, **running_params)
        running_results[name] = time.time() - start_time
        print()

    full_pdp_df_length = build_points_for_full_pdp(data, model).shape[0]
    print(f"The legnth of the consumer data in the full PDP run is {full_pdp_df_length}")

    print("running times:")
    for name in running_results:
        print(f"{name} took: {running_results[name]} sec")

In [ ]:
messure_woodelf_pdp_times(gradient_boosting_model, data=fraud_train, params=get_pdp_testing_params())

Exact PDP k=5:
Building the points took: 2.678842067718506 sec


Preprocessing the trees: 100%|██████████| 100/100 [00:04<00:00, 20.03it/s]


cache misses: 49, cache used: 4406


Computing the values: 100%|██████████| 100/100 [00:03<00:00, 33.07it/s]



Exact PDP k=10:
Building the points took: 2.715407609939575 sec


...


Estimated PDP k=5:
Building the points took: 2.699838161468506 sec


Preprocessing the trees: 100%|██████████| 100/100 [00:00<00:00, 320.41it/s]


cache misses: 49, cache used: 4406


Computing the values: 100%|██████████| 100/100 [00:02<00:00, 44.36it/s]



The legnth of the consumer data in the full PDP run is 66
running times:
Exact PDP k=5 took: 13.116366386413574 sec
Exact PDP k=10 took: 10.048194885253906 sec
Exact PDP k=100 took: 9.09599781036377 sec
Exact full PDP took: 6.965677738189697 sec
Estimated PDP k=5 took: 5.387517690658569 sec


In [ ]:
start_time = time.time()
woodelf_pdp_joint(gradient_boosting_model, fraud_train, k = 5, accurate = True, centered=False, GPU = True)
print(f"\n Exact Joint PDP k=5 computation took: {time.time() - start_time} sec")

Building the points took: 2.8960769176483154 sec. The size of the created df 225


Preprocessing the trees: 100%|██████████| 100/100 [00:03<00:00, 33.02it/s]


cache misses: 49, cache used: 4406


Computing the values: 100%|██████████| 100/100 [00:06<00:00, 16.23it/s]



 Exact Joint PDP k=5 computation took: 25.01784086227417 sec


In [ ]:
start_time = time.time()
woodelf_pdp_joint(gradient_boosting_model, fraud_train, k = 5, accurate = False, centered=False, GPU = True)
print(f"\n Estimated Joint PDP k=5 computation took: {time.time() - start_time} sec")

Building the points took: 2.7417986392974854 sec. The size of the created df 225


Preprocessing the trees: 100%|██████████| 100/100 [00:00<00:00, 121.46it/s]


cache misses: 49, cache used: 4406


Computing the values: 100%|██████████| 100/100 [00:05<00:00, 16.86it/s]



 Estimated Joint PDP k=5 computation took: 21.381614208221436 sec


In [ ]:
start_time = time.time()
woodelf.calculate_background_metric(
    gradient_boosting_model, consumer_data=fraud_train.head(10_000), background_data=fraud_train, metric=PDIV(), global_importance=False, GPU=True
)
print(f"\n Any Order PDIVs n=10,000 took: {time.time() - start_time} sec")

Preprocessing the trees: 100%|██████████| 100/100 [00:04<00:00, 21.04it/s]


cache misses: 49, cache used: 4406


Computing the values: 100%|██████████| 100/100 [00:17<00:00,  5.73it/s]


 Any Order PDIVs n=10,000 took: 22.56028437614441 sec


In [ ]:
start_time = time.time()
woodelf.calculate_background_metric(
    gradient_boosting_model, consumer_data=fraud_train, background_data=fraud_train, metric=PDIV(), global_importance=True, GPU=True
)
print(f"\n Any Order PDIVs took: {time.time() - start_time} sec")

Preprocessing the trees: 100%|██████████| 100/100 [00:04<00:00, 20.77it/s]


cache misses: 49, cache used: 4406


Computing the values: 100%|██████████| 100/100 [00:25<00:00,  3.95it/s]


 Any Order PDIVs took: 30.65900421142578 sec


# KDD-Cup 1999: Intrusion Detection Dataset

In [ ]:
detection_data = pd.read_parquet("drive/MyDrive/PATH_TO_DATA_CREATED_IN_THE_Preprocess_Data_NOTEBOOK.parquet")
unlabeled_data = pd.read_parquet("drive/MyDrive/PATH_TO_DATA_CREATED_IN_THE_Preprocess_Data_NOTEBOOK.parquet")
small_test_data = pd.read_parquet("drive/MyDrive/PATH_TO_DATA_CREATED_IN_THE_Preprocess_Data_NOTEBOOK.parquet")

detection_train_features_names = [f for f in detection_data.columns if f != "target"]
detection_trainset = detection_data[detection_train_features_names]

In [ ]:
detection_model = sklearn.ensemble.HistGradientBoostingRegressor(
    max_iter=100,
    max_depth=6,
    max_leaf_nodes=None,
    # tree_method="hist",  # use 'gpu_hist' if you want GPU support
    random_state=42
)
detection_model.fit(detection_trainset, detection_data['target'])

y_pred = detection_model.predict(small_test_data[detection_train_features_names])
print(f"Accuracy: {accuracy_score(small_test_data['target'], y_pred.round())}, F1 score: {f1_score(small_test_data['target'], y_pred.round())}")

Accuracy: 0.9264859546858975, F1 score: 0.9522245415206658


## Woodelf PDP computation


In [ ]:
messure_woodelf_pdp_times(detection_model, data=detection_trainset, params=get_pdp_testing_params())

Exact PDP k=5:
Building the points took: 6.687107086181641 sec


Preprocessing the trees: 100%|██████████| 100/100 [00:11<00:00,  8.79it/s]


cache misses: 74, cache used: 3484


Computing the values: 100%|██████████| 100/100 [00:01<00:00, 57.21it/s]



Exact PDP k=10:
Building the points took: 6.69488787651062 sec


...


Estimated PDP k=5:
Building the points took: 6.613479375839233 sec


Preprocessing the trees: 100%|██████████| 100/100 [00:00<00:00, 383.52it/s]


cache misses: 74, cache used: 3484


Computing the values: 100%|██████████| 100/100 [00:01<00:00, 55.94it/s]


The legnth of the consumer data in the full PDP run is 44
running times:
Exact PDP k=5 took: 33.43604588508606 sec
Exact PDP k=10 took: 33.48807144165039 sec
Exact PDP k=100 took: 33.61421298980713 sec
Exact full PDP took: 26.75858211517334 sec
Estimated PDP k=5 took: 8.742356300354004 sec


In [ ]:
start_time = time.time()
joint_pdvs = woodelf_pdp_joint(detection_model, detection_trainset, k = 5, accurate = True, centered=False, GPU = True)
print(f"\n Exact Joint PDP k=5 computation took: {time.time() - start_time} sec")

Building the points took: 6.620568752288818 sec. The size of the created df 175


Preprocessing the trees: 100%|██████████| 100/100 [00:11<00:00,  8.46it/s]


cache misses: 74, cache used: 3484


Computing the values: 100%|██████████| 100/100 [00:04<00:00, 21.28it/s]



 Exact Joint PDP k=5 computation took: 37.507641315460205 sec


In [ ]:
start_time = time.time()
joint_pdvs = woodelf_pdp_joint(detection_model, detection_trainset, k = 5, accurate = False, centered=False, GPU = True)
print(f"\n Estimated Joint PDP k=5 computation took: {time.time() - start_time} sec")

Building the points took: 6.6728925704956055 sec. The size of the created df 175


Preprocessing the trees: 100%|██████████| 100/100 [00:00<00:00, 146.68it/s]


cache misses: 74, cache used: 3484


Computing the values: 100%|██████████| 100/100 [00:04<00:00, 21.68it/s]



 Estimated Joint PDP k=5 computation took: 26.024979829788208 sec


In [ ]:
start_time = time.time()
woodelf.calculate_background_metric(
    detection_model, consumer_data=detection_trainset.head(10_000), background_data=detection_trainset, metric=PDIV(), global_importance=False, GPU=True
)
print(f"Any Order PDIVs n=10,000 took: {time.time() - start_time} sec")

Preprocessing the trees: 100%|██████████| 100/100 [00:12<00:00,  7.91it/s]


cache misses: 74, cache used: 3484


Computing the values: 100%|██████████| 100/100 [00:11<00:00,  9.05it/s]

Any Order PDIVs n=10,000 took: 24.306450605392456 sec


In [ ]:
# This cell requires the L4 runtime type (takes 20GB GPU RAM). I run it seperatly.

start_time = time.time()
woodelf.calculate_background_metric(
    detection_model, consumer_data=detection_trainset, background_data=detection_trainset, metric=PDIV(), global_importance=True, GPU=True
)
print(f"Any Order PDIVs took: {time.time() - start_time} sec")

Preprocessing the trees: 100%|██████████| 100/100 [00:12<00:00,  8.13it/s]


cache misses: 74, cache used: 3484


Computing the values: 100%|██████████| 100/100 [00:30<00:00,  3.24it/s]

Any Order PDIVs took: 44.57660126686096 sec
